# More Than A Hack 2027 — Posesión del balón
Objetivo: etiquetar cada frame (H/A/D/X) y superar el baseline (spellF1 0.2433).

## 0. Setup y datos
Verificamos que el notebook encuentra el repo y los 4 archivos del táctico.


Primero revisamos que el notebook encuentre el repo y los archivos del partido. Los datos viven fuera del repo, en la carpeta que apunta `MTAH_DATA_DIR`, para no subirlos a GitHub por accidente. Aquí no usamos el video, solo el tracking (ATD) y el Smart Tagging.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "evaluation"))

from games import asset_paths

print("MTAH_DATA_DIR:", os.environ.get("MTAH_DATA_DIR"))
for name, path in asset_paths().items():
    print(f"{name:<15} {'OK' if Path(path).exists() else 'FALTA'}  {Path(path).name}")

## 1. Cargar el tracking (ATD)
Cada línea del archivo es un frame: `frame_id : 46 slots : balón`, y cada objeto trae `x,y,velocidad` (NaN si no se detectó). Un *slot* es un casillero del tracker, no una persona: hay 46 para unos 22 jugadores, y el mismo jugador puede cambiar de slot.

Lo pasamos a dos arreglos en metros: `jug` (199319, 46, 3) y `bal` (199319, 3). El balón aparece en el 56.9% de los frames (59.3% de los calificados) y hay unos 20.5 slots activos por frame. Guardamos un caché `.npz` fuera del repo para no releer el texto cada vez.

In [ ]:
import xml.etree.ElementTree as ET

rutas = asset_paths()
raiz = ET.parse(rutas["atd_meta"]).getroot()

equipos = [(t.get("id"), t.findtext("Name")) for t in raiz.iter("Team")]
print("Equipos:", equipos)

slots = [p.get("teamId") for p in raiz.iter("Player")]
print("Total de slots:", len(slots))
for id_equipo, nombre in equipos:
    print(nombre, "->", slots.count(id_equipo), "slots")

In [ ]:
with open(rutas["atd_raw"]) as f:
    lineas = f.readlines()
print("Líneas (frames):", len(lineas))

frame, jugadores, balon = lineas[0].split(":")
print("Id del primer frame:", frame)
print("Slots en esa línea:", len(jugadores.split(";")))

## 2. Cargar el ATD
Pasamos el texto a arreglos en metros: `jug` (frame, slot, [x, y, vel]) y `bal` (frame, [x, y, vel]).
Lo que no se detectó queda como NaN.

In [ ]:
import numpy as np
from games import GAME, GAMES

N_FRAMES, N_SLOTS = 199319, 46
LARGO, ANCHO = GAMES[GAME]["pitch"]          # 105.3 x 68.0 m

def a_numeros(texto):
    """'x,y,vel' -> (x, y, vel). Si no hay dato -> (nan, nan, nan)."""
    if texto[0] == "N":
        return (np.nan, np.nan, np.nan)
    partes = texto.split(",")
    return (float(partes[0]), float(partes[1]), float(partes[2]))

cache = Path(rutas["atd_raw"]).parent / "atd_arrays.npz"

if cache.exists():
    d = np.load(cache)
    jug, bal = d["jug"], d["bal"]
else:
    jug = np.full((N_FRAMES, N_SLOTS, 3), np.nan, dtype=np.float32)
    bal = np.full((N_FRAMES, 3), np.nan, dtype=np.float32)
    with open(rutas["atd_raw"]) as f:
        for linea in f:
            frame, jugadores, balon = linea.strip().split(":")
            i = int(frame) - 1                    # frame_id empieza en 1
            for s, grupo in enumerate(jugadores.split(";")):
                jug[i, s] = a_numeros(grupo)
            bal[i] = a_numeros(balon)
    # de fracción de cancha (0-1) a metros
    jug[:, :, 0] *= LARGO; jug[:, :, 1] *= ANCHO
    bal[:, 0] *= LARGO;    bal[:, 1] *= ANCHO
    np.savez(cache, jug=jug, bal=bal)

print("jug:", jug.shape, "| bal:", bal.shape)

In [ ]:
# Frames calificados (los únicos que cuentan), tomados de DATA.md
calificado = np.zeros(N_FRAMES, dtype=bool)
calificado[11185:90652] = True      # 1er tiempo
calificado[110808:190198] = True    # 2do tiempo

con_balon = ~np.isnan(bal[:, 0])
n_activos = (~np.isnan(jug[:, :, 0])).sum(axis=1)

print("Balón presente, todos los frames:   ", con_balon.mean().round(3))
print("Balón presente, frames calificados: ", con_balon[calificado].mean().round(3))
print("Slots activos por frame (calificados):", n_activos[calificado].mean().round(1))

## 2. Huecos del balón
Un hueco es una racha seguida de frames sin balón. Hay 150, con una mediana de 10.7 s y uno de hasta 221 s. Casi no hay huecos cortos: el 85% de los frames sin balón está en huecos de más de 10 s.

Por eso no vale la pena interpolar: no hay un balón "casi visible" que rellenar. Hay que inferir quién tiene la posesión usando el contexto.

In [ ]:
sin_balon = ~con_balon & calificado     # sin balón y dentro del tiempo calificado

def huecos(falta, inicio, fin):
    """Inicio y largo (en frames) de cada racha sin balón dentro de [inicio, fin)."""
    x = falta[inicio:fin].astype(np.int8)
    cambios = np.diff(np.concatenate(([0], x, [0])))
    comienzos = np.flatnonzero(cambios == 1)     # 0 -> 1: empieza un hueco
    finales = np.flatnonzero(cambios == -1)      # 1 -> 0: termina
    return comienzos + inicio, finales - comienzos

ini1, largo1 = huecos(sin_balon, 11185, 90652)       # 1er tiempo
ini2, largo2 = huecos(sin_balon, 110808, 190198)     # 2do tiempo
inicios = np.concatenate([ini1, ini2])
largos = np.concatenate([largo1, largo2])
seg = largos / 25

print("Frames sin balón (suma de huecos):", largos.sum(), "| directo:", sin_balon.sum())
print("Cantidad de huecos:", len(seg))
print("Mediana (s):", np.median(seg).round(2))
print("Percentiles 75 / 90 / 99 (s):", np.percentile(seg, [75, 90, 99]).round(1))
print("Hueco más largo (s):", seg.max().round(1))

In [ ]:
for corte in [0.5, 1, 3, 10]:
    pct = largos[seg > corte].sum() / largos.sum()
    print(f"Frames perdidos en huecos de más de {corte} s: {pct:.1%}")

## 3. ¿Qué pasa cuando el balón desaparece?
Comparamos el juego con y sin balón visible. Sin balón los jugadores van más lentos (1.17 contra 2.16 m/s) y se detectan menos slots (19.2 contra 21.4). Pero las distribuciones por tramo se pisan, así que no hay un corte limpio entre "juego" y "pausa".

Conclusión: la velocidad sirve como pista, no como regla para decidir balón muerto. También confirmamos que nunca falta todo el equipo en el mismo frame, y que 429 frames traen posición pero no velocidad.

In [ ]:
falta_c = sin_balon[calificado]                    # sin balón, solo frames calificados
vel_jug = np.nanmean(jug[calificado, :, 2], axis=1)  # velocidad media de jugadores (m/s)
activos = n_activos[calificado]

print("Velocidad media de jugadores (m/s)")
print("  con balón visible:", vel_jug[~falta_c].mean().round(2))
print("  sin balón        :", vel_jug[falta_c].mean().round(2))
print("\nSlots activos")
print("  con balón visible:", activos[~falta_c].mean().round(1))
print("  sin balón        :", activos[falta_c].mean().round(1))
print("\n% de frames sin balón por tiempo")
print("  1er tiempo:", (largo1.sum() / 79467).round(3), "| 2do tiempo:", (largo2.sum() / 79390).round(3))

In [ ]:
vacio = (n_activos == 0) & calificado      # frames calificados sin ningún jugador

print("Frames calificados sin ningún jugador:", vacio.sum(), f"({vacio[calificado].mean():.1%})")
print("  de ellos, también sin balón:", (vacio & sin_balon).sum())

v1 = huecos(vacio, 11185, 90652)[1]
v2 = huecos(vacio, 110808, 190198)[1]
largos_vacio = np.concatenate([v1, v2]) / 25
if len(largos_vacio):
    print("Rachas vacías:", len(largos_vacio),
          "| mediana (s):", np.median(largos_vacio).round(1),
          "| máxima (s):", largos_vacio.max().round(1))

In [ ]:
hay_jug = activos > 0

print("Velocidad media (m/s), solo frames con jugadores")
print("  con balón visible:", vel_jug[~falta_c & hay_jug].mean().round(2))
print("  sin balón        :", vel_jug[falta_c & hay_jug].mean().round(2))
print("\nSlots activos, solo frames con jugadores")
print("  con balón visible:", activos[~falta_c & hay_jug].mean().round(1))
print("  sin balón        :", activos[falta_c & hay_jug].mean().round(1))

In [ ]:
n_vel = (~np.isnan(jug[:, :, 2])).sum(axis=1)               # slots con velocidad, por frame

sin_vel = (n_vel == 0) & (n_activos > 0) & calificado        # hay posiciones, ninguna velocidad
print("Frames con posición pero sin velocidad:", sin_vel.sum(), f"({sin_vel[calificado].mean():.1%} de los calificados)")
print("  de ellos, también sin balón:", (sin_vel & sin_balon).sum())

r1 = huecos(sin_vel, 11185, 90652)[1]
r2 = huecos(sin_vel, 110808, 190198)[1]
rachas = np.concatenate([r1, r2]) / 25
if len(rachas):
    print("Rachas:", len(rachas), "| mediana (s):", np.median(rachas).round(1),
          "| máxima (s):", rachas.max().round(1))

In [ ]:
hay_vel = (n_vel > 0)[calificado]

print("Velocidad media de jugadores (m/s), solo frames con velocidad")
print("  con balón visible:", vel_jug[~falta_c & hay_vel].mean().round(2))
print("  sin balón        :", vel_jug[falta_c & hay_vel].mean().round(2))

## 5. Velocidad de jugadores por tramo
Comparamos la velocidad media de jugadores en cada hueco del balón contra cada tramo con balón visible.
Si los huecos forman dos grupos, una parte son pausas y otra es juego vivo.

In [ ]:
# Velocidad media de jugadores por frame, en la línea de tiempo completa
vel_full = np.full(N_FRAMES, np.nan)
vel_full[calificado] = vel_jug

def vel_por_tramo(inicios, largos):
    """Velocidad media de jugadores dentro de cada tramo."""
    return np.array([np.nanmean(vel_full[i:i + l]) for i, l in zip(inicios, largos)])

# Tramos con balón visible (mismo método que los huecos, pero al revés)
con_c = con_balon & calificado
vi1, vl1 = huecos(con_c, 11185, 90652)
vi2, vl2 = huecos(con_c, 110808, 190198)
vis_ini = np.concatenate([vi1, vi2])
vis_largo = np.concatenate([vl1, vl2])

v_hueco = vel_por_tramo(inicios, largos)
v_visible = vel_por_tramo(vis_ini, vis_largo)

print("Tramos con balón visible:", len(v_visible), "| huecos:", len(v_hueco))
print("\nVelocidad media por tramo (m/s), percentiles 10 / 25 / 50 / 75 / 90")
print("  con balón visible:", np.nanpercentile(v_visible, [10, 25, 50, 75, 90]).round(2))
print("  huecos           :", np.nanpercentile(v_hueco, [10, 25, 50, 75, 90]).round(2))

In [ ]:
import matplotlib.pyplot as plt

plt.hist(v_visible, bins=30, alpha=0.6, label="balón visible")
plt.hist(v_hueco, bins=30, alpha=0.6, label="huecos")
plt.xlabel("velocidad media de jugadores (m/s)")
plt.ylabel("número de tramos")
plt.legend()
plt.show()

### Smart Tagging
Son 638 fases etiquetadas automáticamente. Las de balón parado duran siempre 35 s (5 s los saques iniciales) y cubren el 27.6% de los frames calificados, o sea que su duración no dice nada. Lo útil es el inicio: ahí la velocidad baja y el balón desaparece (visible solo 25-35% del tiempo), porque el juego se detiene.

Lo usaremos como ancla de "aquí se reinicia el juego", sin copiar las fases como etiquetas.

## 6. Smart Tagging vs. huecos del balón
Marcamos qué frames caen dentro de alguna fase y cuáles dentro de una jugada a balón parado,
y comparamos los frames con y sin balón visible.

In [ ]:
from starter.load_smart_tagging import load as cargar_tags

tags = cargar_tags()
print("Fases cargadas:", len(tags))

# Máscaras por frame: dentro de alguna fase / dentro de una jugada a balón parado
en_fase = np.zeros(N_FRAMES, dtype=bool)
en_parado = np.zeros(N_FRAMES, dtype=bool)
for t in tags:
    en_fase[t["start_frame"]:t["end_frame"]] = True
    if (t["code"] or "").strip().lower() == "set pieces":
        en_parado[t["start_frame"]:t["end_frame"]] = True

grupos = [("todos los calificados", calificado),
          ("con balón visible", con_balon & calificado),
          ("sin balón (huecos)", sin_balon)]

for nombre, mascara in grupos:
    print(nombre)
    print("   dentro de alguna fase     :", en_fase[mascara].mean().round(3))
    print("   dentro de balón parado    :", en_parado[mascara].mean().round(3))

In [ ]:
print("Velocidad media de jugadores (m/s)")
print("  en balón parado :", np.nanmean(vel_full[en_parado & calificado]).round(2))
print("  fuera de balón parado:", np.nanmean(vel_full[~en_parado & calificado]).round(2))

## 7. Anatomía de una jugada a balón parado
Duración de las fases por tipo, y el promedio de velocidad de jugadores y de visibilidad del balón
alrededor del inicio de cada fase (de -10 s a +30 s).

In [ ]:
dur_por_tipo = {}
for t in tags:
    if (t["code"] or "").strip().lower() == "set pieces":
        tipo = t["tags"].get("Type")
        dur_por_tipo.setdefault(tipo, []).append((t["end_frame"] - t["start_frame"]) / 25)

for tipo, d in dur_por_tipo.items():
    print(f"{tipo:<24} n={len(d):<3} mediana={np.median(d):5.1f} s   máx={max(d):5.1f} s")

In [ ]:
ventana = np.arange(-10 * 25, 30 * 25)       # de -10 s a +30 s, en frames
inicios_parado = [t["start_frame"] for t in tags if (t["code"] or "").strip().lower() == "set pieces"]

curvas_vel, curvas_balon = [], []
for f in inicios_parado:
    idx = f + ventana
    if idx.min() >= 0 and idx.max() < N_FRAMES:
        curvas_vel.append(vel_full[idx])
        curvas_balon.append(con_balon[idx])

fig, ejes = plt.subplots(2, 1, sharex=True)
ejes[0].plot(ventana / 25, np.nanmean(curvas_vel, axis=0))
ejes[0].set_ylabel("velocidad jugadores (m/s)")
ejes[1].plot(ventana / 25, np.mean(curvas_balon, axis=0))
ejes[1].set_ylabel("fracción con balón visible")
ejes[1].set_xlabel("segundos desde el inicio de la fase de balón parado")
for e in ejes:
    e.axvline(0, color="gray", linestyle="--")
plt.show()

## Hallazgos de la auditoría
- El ATD está alineado: 199,319 frames, `video_frame = frame_id - 1`. Balón presente en 59.3% de los calificados.
- Hay 150 huecos del balón, mediana 10.7 s, el más largo 221 s. Interpolar no sirve.
- Durante los huecos los jugadores van más lento (1.17 vs 2.16 m/s) y hay ~2 slots menos activos.
- La velocidad sola no separa pausa de juego vivo (las distribuciones se superponen).
- Las fases de balón parado del Smart Tagging son ventanas fijas de 35 s: solo su inicio sirve.
- Alrededor del inicio, la velocidad baja y el balón desaparece: pausa real que coincide con el hueco.

### Reproducir el baseline
Antes de mejorar el baseline, comprobamos que lo entendemos: asignamos cada frame al equipo del jugador más cercano al balón (y D si no hay balón), pero vectorizado. Coincide 100% con `baseline_raw.csv`, así que la lectura de datos, la conversión a metros y los equipos están bien.

## 8. Reproducir el baseline_raw con nuestros arreglos
Distancia del jugador más cercano de cada equipo al balón, y la etiqueta H/A/D resultante.
Debe coincidir con `evaluation/results/baseline_raw.csv`.

In [ ]:
import pandas as pd

# Equipo de cada slot: 0 = local (primer equipo del XML), 1 = visitante
id_local = equipos[0][0]
equipo_slot = np.array([0 if s == id_local else 1 for s in slots])

# Distancia (m) de cada slot al balón; sin dato -> infinito
dx = jug[:, :, 0] - bal[:, [0]]
dy = jug[:, :, 1] - bal[:, [1]]
dist = np.nan_to_num(np.sqrt(dx**2 + dy**2), nan=np.inf)

# Jugador más cercano de cada equipo
dist_local = np.min(np.where(equipo_slot == 0, dist, np.inf), axis=1)
dist_visit = np.min(np.where(equipo_slot == 1, dist, np.inf), axis=1)

# Etiqueta: equipo del más cercano; D si no hay balón (o no hay jugadores)
hay_dato = np.isfinite(np.minimum(dist_local, dist_visit))
etiqueta = np.where(hay_dato, np.where(dist_local <= dist_visit, "H", "A"), "D")

ref = pd.read_csv(ROOT / "evaluation" / "results" / "baseline_raw.csv")["label"].values
print("Coincidencia con baseline_raw, todos los frames:", (etiqueta == ref).mean().round(5))
print("Coincidencia en frames calificados:", (etiqueta[calificado] == ref[calificado]).mean().round(5))

## 9. Distancia del jugador más cercano
Con balón visible: ¿a qué distancia está el jugador más cercano, y qué tan seguido hay duelo
(los más cercanos de cada equipo casi a la misma distancia)?

In [ ]:
vis = con_balon & calificado
cerca = np.minimum(dist_local, dist_visit)          # distancia del más cercano (m)
margen = np.abs(dist_local - dist_visit)            # diferencia entre el más cercano de cada equipo

print("Distancia del jugador más cercano al balón (m), con balón visible")
print("  percentiles 10 / 25 / 50 / 75 / 90 / 99:", np.percentile(cerca[vis], [10, 25, 50, 75, 90, 99]).round(1))
for lim in [1, 2, 3, 5]:
    print(f"  más cercano a menos de {lim} m: {(cerca[vis] < lim).mean():.1%}")

print(f"\nFrames en duelo (diferencia < 2 m entre equipos): {(margen[vis] < 2).mean():.1%}")

vel_bal = bal[:, 2]
print("\nVelocidad del balón (m/s), percentiles 10 / 25 / 50 / 75 / 90 / 99:",
      np.nanpercentile(vel_bal[vis], [10, 25, 50, 75, 90, 99]).round(1))

In [ ]:
plt.hist(cerca[vis & (cerca < 30)], bins=60)
plt.xlabel("distancia del jugador más cercano al balón (m)")
plt.ylabel("número de frames")
plt.show()

## 10. Velocidad del balón y zonas de certeza
La velocidad del archivo es unas 4 veces menor que la calculada desde las posiciones (mediana 1.3 contra 5.1 m/s), y no es solo un tema de unidades. Preferimos calcularla nosotros, pero con una ventana de varios frames, porque la posición del balón es ruidosa.

Con eso dividimos los frames calificados por certeza: control claro 23.1%, balón visible pero dudoso 36.2% y sin balón 40.7%.

In [ ]:
# Velocidad calculada: metros recorridos entre dos frames consecutivos con balón, por 25 fps
ambos = con_balon[1:] & con_balon[:-1] & calificado[1:] & calificado[:-1]
paso = np.sqrt((bal[1:, 0] - bal[:-1, 0])**2 + (bal[1:, 1] - bal[:-1, 1])**2)   # metros por frame
vel_calc = paso * 25

print("Velocidad del balón (m/s), percentiles 10 / 25 / 50 / 75 / 90 / 99")
print("  del archivo :", np.nanpercentile(vel_bal[1:][ambos], [10, 25, 50, 75, 90, 99]).round(1))
print("  calculada   :", np.nanpercentile(vel_calc[ambos], [10, 25, 50, 75, 90, 99]).round(1))

In [ ]:
control_claro = vis & (cerca < 2) & (margen >= 2)    # más cercano a menos de 2 m y sin duelo

print("Reparto de los frames calificados")
print(f"  control claro        : {control_claro[calificado].mean():.1%}")
print(f"  balón visible dudoso : {(vis & ~control_claro)[calificado].mean():.1%}")
print(f"  sin balón            : {sin_balon[calificado].mean():.1%}")

## 11. Esqueleto de posesión
Los frames de control claro son semillas: sabemos qué equipo tiene el balón.
Medimos cada cuánto aparecen y cuántas veces cambia el equipo entre semillas consecutivas.

### Semillas: los frames donde sí sabemos
Llamamos *semillas* a los frames de control claro, donde el equipo con la posesión es casi seguro. Son 36,775, pero vienen en ráfagas: el 51.5% del partido cae en tramos de más de 5 s sin ninguna semilla. Además hay 487 cambios de equipo entre semillas, 221 de menos de 1 s (probablemente ruido o duelos).

Las semillas son buenas anclas, pero la mitad del partido hay que inferirla.

In [ ]:
equipo_cercano = np.where(dist_local <= dist_visit, 0, 1)      # 0 = local, 1 = visitante

idx = np.flatnonzero(control_claro & calificado)               # frames semilla
equipo = equipo_cercano[idx]
misma_mitad = (idx[1:] >= 110808) == (idx[:-1] >= 110808)      # no cruzar el medio tiempo

dt = np.diff(idx) / 25                                         # segundos entre semillas consecutivas
cambia = equipo[1:] != equipo[:-1]                             # ¿la siguiente semilla es del otro equipo?
dt, cambia = dt[misma_mitad], cambia[misma_mitad]

print("Frames semilla:", len(idx))
print("Segundos hasta la siguiente semilla, percentiles 90 / 99:", np.percentile(dt, [90, 99]).round(1))
print(f"Tiempo en tramos de más de 5 s sin semilla : {dt[dt > 5].sum() / dt.sum():.1%}")
print(f"Tiempo en tramos de más de 10 s sin semilla: {dt[dt > 10].sum() / dt.sum():.1%}")

print("\nCambios de equipo entre semillas consecutivas:", cambia.sum())
print("  con menos de 1 s entre semillas:", (cambia & (dt < 1)).sum())
print("  con más de 5 s entre semillas  :", (cambia & (dt > 5)).sum())

## 12. Orientar la cancha
Para cada equipo y tiempo, buscamos de qué lado está su portero (el jugador más atrasado)
para saber hacia dónde ataca. En el 2do tiempo se invierte.

### Hacia dónde ataca cada equipo
Para que "adelante" signifique lo mismo siempre, hay que saber hacia qué portería ataca cada equipo en cada tiempo. Lo sacamos del portero: es el jugador más atrasado. El local ataca hacia +x en el primer tiempo y hacia -x en el segundo, y el visitante al revés. Con eso orientamos las coordenadas de cada equipo hacia la portería rival.

In [ ]:
mitades = {"1er tiempo": slice(11185, 90652), "2do tiempo": slice(110808, 190198)}

for nombre, sl in mitades.items():
    for eq, etiqueta in [(0, "local"), (1, "visitante")]:
        x = jug[sl, :, 0][::25][:, equipo_slot == eq]      # 1 frame por segundo, solo ese equipo
        x_min = np.nanmin(x, axis=1).mean()                # promedio del jugador más cercano a x = 0
        x_max = np.nanmax(x, axis=1).mean()                # promedio del más cercano a x = largo
        ataca_derecha = x_min < (LARGO - x_max)            # portero más cerca de x = 0
        print(f"{nombre} | {etiqueta:<9} min medio {x_min:5.1f} m | max medio {x_max:5.1f} m | "
              f"ataca hacia {'derecha (+x)' if ataca_derecha else 'izquierda (-x)'}")

In [ ]:
for nombre, sl in mitades.items():
    for eq, etiqueta in [(0, "local"), (1, "visitante")]:
        x = jug[sl, :, 0][::25][:, equipo_slot == eq]
        vacios = np.isnan(x).all(axis=1)                   # segundos sin ningún jugador de ese equipo
        x = x[~vacios]
        x_min = np.nanmin(x, axis=1).mean()
        x_max = np.nanmax(x, axis=1).mean()
        ataca_derecha = x_min < (LARGO - x_max)
        print(f"{nombre} | {etiqueta:<9} min {x_min:5.1f} m | max {x_max:5.1f} m | "
              f"ataca {'derecha (+x)' if ataca_derecha else 'izquierda (-x)'} | segundos sin datos: {vacios.sum()}")

# ¿En cuántos frames calificados un equipo entero no tiene ningún jugador con dato?
hay = ~np.isnan(jug[:, :, 0])
act_local = hay[:, equipo_slot == 0].sum(axis=1)
act_visit = hay[:, equipo_slot == 1].sum(axis=1)
print("\nFrames calificados sin ningún jugador local    :", (calificado & (act_local == 0)).sum())
print("Frames calificados sin ningún jugador visitante:", (calificado & (act_visit == 0)).sum())

## 13. Faltantes disfrazados
¿Hay posiciones fuera de la cancha, en (0, 0) o congeladas? ¿Cuándo no hay jugadores locales?

In [ ]:
def revisar(nombre, x, y):
    x, y = x[calificado], y[calificado]
    fuera = (x < 0) | (x > LARGO) | (y < 0) | (y > ANCHO)
    cero = (x == 0) & (y == 0)
    print(f"{nombre:<10} fuera de la cancha: {fuera.sum():>7} | exactamente (0, 0): {cero.sum():>5}")

revisar("jugadores", jug[:, :, 0], jug[:, :, 1])
revisar("balón", bal[:, 0], bal[:, 1])

In [ ]:
# ¿Misma posición exacta en dos frames consecutivos? En el balón o en un jugador en movimiento es raro
mismo_bal = (bal[1:, 0] == bal[:-1, 0]) & (bal[1:, 1] == bal[:-1, 1])
print(f"Balón: pasos de 0 m exactos: {mismo_bal[ambos].mean():.2%}")

mismo_jug = (jug[1:, :, 0] == jug[:-1, :, 0]) & (jug[1:, :, 1] == jug[:-1, :, 1])
ambos_jug = hay[1:] & hay[:-1] & calificado[1:, None] & calificado[:-1, None]
print(f"Jugadores: pasos de 0 m exactos: {mismo_jug[ambos_jug].mean():.2%}")

In [ ]:
sin_local = calificado & (act_local == 0)
largos_l = np.concatenate([huecos(sin_local, 11185, 90652)[1],
                           huecos(sin_local, 110808, 190198)[1]]) / 25

print("Rachas sin ningún jugador local:", len(largos_l))
print("  duración en segundos (mín / mediana / máx):",
      largos_l.min().round(1), np.median(largos_l).round(1), largos_l.max().round(1))
print(f"  con balón visible en esos frames: {con_balon[sin_local].mean():.1%}")

## 14. Qué tan fuera de la cancha
Metros que se salen de la cancha jugadores y balón. Y cuántas veces el "más cercano" al balón
es un objeto fuera de la cancha. También quitamos de las semillas los frames donde falta un equipo.

In [ ]:
def exceso(x, y):
    """Metros que se sale de la cancha (0 si está dentro)."""
    dx = np.maximum(np.maximum(-x, x - LARGO), 0)
    dy = np.maximum(np.maximum(-y, y - ANCHO), 0)
    return np.hypot(dx, dy)

e_jug = exceso(jug[:, :, 0], jug[:, :, 1])          # (frames, slots)
e_bal = exceso(bal[:, 0], bal[:, 1])

fuera_j = e_jug[calificado][e_jug[calificado] > 0]
fuera_b = e_bal[calificado][e_bal[calificado] > 0]

print("Jugadores fuera de la cancha: percentiles 50 / 90 / 99 del exceso (m):", np.percentile(fuera_j, [50, 90, 99]).round(1))
print(f"  más de 1 m: {(fuera_j > 1).mean():.1%} | más de 3 m: {(fuera_j > 3).mean():.1%} | más de 10 m: {(fuera_j > 10).mean():.1%}")
print("Balón fuera de la cancha: percentiles 50 / 90 / 99 del exceso (m):", np.percentile(fuera_b, [50, 90, 99]).round(1))
print(f"  más de 0.5 m: {(fuera_b > 0.5).mean():.1%} | más de 1 m: {(fuera_b > 1).mean():.1%} | más de 3 m: {(fuera_b > 3).mean():.1%}")

In [ ]:
cercano = np.argmin(dist, axis=1)                                   # slot más cercano al balón
fuera_cercano = e_jug[np.arange(N_FRAMES), cercano] > 1             # a más de 1 m fuera de la cancha
print(f"Frames con balón visible donde el más cercano está a más de 1 m fuera de la cancha: {fuera_cercano[vis].mean():.2%}")

In [ ]:
tracking_ok = (act_local > 0) & (act_visit > 0)
antes = control_claro.sum()
control_claro = control_claro & tracking_ok
print("Semillas antes:", antes, "| después:", control_claro.sum())

## Decisiones de la auditoría
- Velocidad del balón: se calcula desde posiciones; la del archivo está subestimada ~4x.
- Semillas (control claro): más cercano a menos de 2 m y sin duelo, solo si ambos equipos tienen jugadores.
- Balón fuera de la cancha (más de 1 m): evidencia fuerte de balón muerto.
- Huecos del balón: no se interpolan; se infieren con contexto (forma de los equipos, velocidad, Smart Tagging).
- Smart Tagging: solo cuenta el inicio de las fases de balón parado (ventanas fijas de 35 s).
- Sin faltantes disfrazados: sin (0, 0) ni valores congelados. Una racha de 30 s sin jugadores locales.
- Orientación: el local ataca a la derecha en el 1er tiempo y a la izquierda en el 2do; el visitante al revés.

## 15. Features de forma de equipo (sin balón)
Orientamos cada equipo hacia su portería rival y resumimos su posición, forma y movimiento.

### Features de forma de equipo (sin balón)
Cuando el balón no se ve, miramos cómo está parado cada equipo. Por equipo y frame, ya orientados hacia su ataque, calculamos qué tan adelantado está (`cx`), cuánto se extiende a lo largo y a lo ancho (`sx`, `sy`), dónde está su línea defensiva (`linea_def`), qué tan rápido se mueve (`vel`) y cuánto avanzó en el último segundo (`dcx_1s`). Son 12 features en total.

Solo hay NaN donde falta un equipo o la velocidad. No los imputamos, porque LightGBM y XGBoost los aceptan.

In [ ]:
import warnings

# Sentido de ataque por equipo y tiempo: +1 ataca hacia +x, -1 hacia -x (calculado con el portero)
sentido = {}
for nombre, sl in mitades.items():
    for eq in (0, 1):
        x = jug[sl, :, 0][::25][:, equipo_slot == eq]
        x = x[~np.isnan(x).all(axis=1)]
        x_min, x_max = np.nanmin(x, axis=1).mean(), np.nanmax(x, axis=1).mean()
        sentido[(nombre, eq)] = 1 if x_min < (LARGO - x_max) else -1
print(sentido)

segunda = np.arange(N_FRAMES) >= 110808            # True en el 2do tiempo
s_loc = np.where(segunda, sentido[("2do tiempo", 0)], sentido[("1er tiempo", 0)])
s_vis = np.where(segunda, sentido[("2do tiempo", 1)], sentido[("1er tiempo", 1)])

In [ ]:
def forma_equipo(eq, signo):
    """Resumen de un equipo por frame, con x orientado hacia su portería rival."""
    x = jug[:, :, 0][:, equipo_slot == eq]
    y = jug[:, :, 1][:, equipo_slot == eq]
    v = jug[:, :, 2][:, equipo_slot == eq]
    x = np.where(signo[:, None] == 1, x, LARGO - x)      # x grande = cerca de la portería rival
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)   # frames sin jugadores de ese equipo -> NaN
        return {"cx": np.nanmean(x, axis=1),
                "sx": np.nanstd(x, axis=1),
                "sy": np.nanstd(y, axis=1),
                "linea_def": np.nanpercentile(x, 20, axis=1),
                "vel": np.nanmean(v, axis=1)}

def cambio_1s(a):
    """Diferencia respecto a 1 segundo (25 frames) antes."""
    out = np.full_like(a, np.nan)
    out[25:] = a[25:] - a[:-25]
    return out

cols = {}
for pref, f in (("L", forma_equipo(0, s_loc)), ("V", forma_equipo(1, s_vis))):
    for nombre, a in f.items():
        cols[f"{pref}_{nombre}"] = a
    cols[f"{pref}_dcx_1s"] = cambio_1s(f["cx"])

feats = pd.DataFrame(cols)
print(feats.shape)
print("\n% de NaN por feature (frames calificados):")
print((feats[calificado].isna().mean() * 100).round(2).to_string())
print("\nAvance medio (cx) de cada equipo:", feats.loc[calificado, ["L_cx", "V_cx"]].mean().round(1).to_dict())

## 16. ¿Se puede inferir el equipo con balón solo desde la forma de los equipos?
Entrenamos sobre las semillas (control claro) con las features sin balón, validando por bloques de tiempo.
Comparamos regresión logística, LightGBM y XGBoost.

### ¿La forma del equipo basta para saber quién tiene la pelota?
Probamos en las semillas si estas 12 features predicen el equipo con la posesión, validando por 6 bloques de tiempo seguidos (se entrena en 5 y se prueba en el que queda).

Sí hay señal: la regresión logística acierta 80.3%, LightGBM 81.1% y XGBoost 81.2%, contra 61.5% de siempre decir "local". LightGBM y XGBoost empatan en la práctica, y los volveremos a comparar con features más ricas. El aviso: las semillas son momentos fáciles, así que el 81% probablemente es optimista para los huecos.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

idx = np.flatnonzero(control_claro)                 # frames semilla
X = feats.iloc[idx].reset_index(drop=True)          # features sin balón
y = equipo_cercano[idx]                             # 0 = local, 1 = visitante

# 6 bloques de tiempo seguidos, con el mismo número de frames calificados cada uno
orden = np.flatnonzero(calificado)
bloque_de_frame = np.full(N_FRAMES, -1)
bloque_de_frame[orden] = np.minimum(np.arange(len(orden)) * 6 // len(orden), 5)
bloque = bloque_de_frame[idx]

print("Semillas:", len(y), "| % del equipo local:", round(1 - y.mean(), 3))
print("Semillas por bloque:", np.bincount(bloque))
print("Referencia (siempre el equipo más frecuente):", round(max(y.mean(), 1 - y.mean()), 3))

In [ ]:
modelos = {
    "Regresión logística": make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=1000)),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, random_state=0,
                               n_jobs=1, deterministic=True, force_row_wise=True, verbose=-1),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, tree_method="hist",
                             random_state=0, n_jobs=1),
}

for nombre, modelo in modelos.items():
    prob = np.zeros(len(y))
    acc_bloques = []
    for b in range(6):
        test = bloque == b
        modelo.fit(X[~test], y[~test])
        prob[test] = modelo.predict_proba(X[test])[:, 1]
        acc_bloques.append(((prob[test] > 0.5) == y[test]).mean())
    acc = ((prob > 0.5) == y).mean()
    print(f"{nombre:<20} exactitud {acc:.3f} | AUC {roc_auc_score(y, prob):.3f} | "
          f"exactitud por bloque: mín {min(acc_bloques):.3f}, máx {max(acc_bloques):.3f}")

## 13. ¿Qué usa el modelo y qué tan bien funciona cerca de los huecos?
Dos chequeos sobre el 81% del paso anterior. Primero, qué features usa el modelo: mezclamos al azar una feature a la vez en el bloque de prueba y vemos cuánto cae el acierto (importancia por permutación). Segundo, si el acierto cae cuando la semilla está cerca de un hueco del balón, que es la situación más parecida a lo que vamos a enfrentar de verdad.

Límite: dentro de un hueco no hay semillas, así que el borde es lo más cercano que podemos medir. Si el modelo aguanta ahí, es buena señal, pero no una prueba para el centro del hueco.

In [ ]:
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

def modelo_xgb():
    return XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                         tree_method="hist", random_state=0, n_jobs=1)

rng = np.random.default_rng(0)
caida = pd.DataFrame(0.0, index=range(6), columns=X.columns)

for b in range(6):
    train, test = bloque != b, bloque == b
    modelo = modelo_xgb().fit(X[train], y[train])
    base = accuracy_score(y[test], modelo.predict(X[test]))
    for col in X.columns:
        caidas = []
        for _ in range(3):                      # 3 mezclas para reducir ruido
            Xp = X[test].copy()
            Xp[col] = rng.permutation(Xp[col].values)
            caidas.append(base - accuracy_score(y[test], modelo.predict(Xp)))
        caida.loc[b, col] = np.mean(caidas)

importancia = (caida.mean() * 100).sort_values(ascending=False)
print("Puntos de acierto que se pierden al mezclar cada feature:")
print(importancia.round(2))

In [ ]:
from scipy.ndimage import distance_transform_edt

# Segundos desde cada frame hasta el frame sin balón más cercano
tiene_balon = con_balon | ~calificado
dist_hueco = distance_transform_edt(tiene_balon) / 25

# Probabilidades fuera de muestra (cada bloque lo predice un modelo que no lo vio)
prob = np.zeros(len(y))
for b in range(6):
    train, test = bloque != b, bloque == b
    prob[test] = modelo_xgb().fit(X[train], y[train]).predict_proba(X[test])[:, 1]

tramos = pd.cut(dist_hueco[idx], [0, 2, 5, 10, np.inf],
                labels=["0-2 s", "2-5 s", "5-10 s", ">10 s"], include_lowest=True)
tabla = (pd.DataFrame({"tramo": tramos, "acierto": (prob > 0.5) == y, "local": y == 0})
         .groupby("tramo", observed=True)
         .agg(semillas=("acierto", "size"), acierto=("acierto", "mean"), pct_local=("local", "mean")))
print(tabla.round(3))

**Resultados.** El ancho del equipo local (`L_sy`) es la feature más usada (12.5 puntos de acierto), seguida de la línea defensiva local, el ancho del visitante y la velocidad de ambos. Los cambios de 1 segundo casi no aportan.

Cerca de un hueco el acierto baja a 73.5%, contra ~84% lejos de ellos. También cambia la composición: lejos de los huecos el local tiene el 71% de las semillas, y cerca solo 46-49%.

**Decisión.** Para los huecos esperamos 70-75%, no 81%. Hay que corregir el sesgo hacia el local y darle más peso al entrenamiento cerca de huecos. Y probar ventanas de tiempo más largas en lugar de cambios de 1 s.

## 14. Contexto temporal y peso por cercanía al hueco
Una posesión dura varios segundos, así que la forma de los equipos en un solo frame es ruidosa. Agregamos el promedio de cada feature en ventanas de 5, 15 y 30 s alrededor del frame (se vale usar frames futuros porque etiquetamos el partido completo, no en vivo).

También probamos darle el mismo peso a cada tramo de cercanía al hueco, porque las semillas lejos de los huecos son las más fáciles y dominaban el entrenamiento. Comparamos 4 configuraciones y nos quedamos con la que mejor funcione cerca de los huecos (tramos 0-2 s y 2-5 s), mirando también el log-loss: las probabilidades van al HMM y tienen que ser creíbles.

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss
from lightgbm import LGBMClassifier

# 1. Promedios en ventanas de 5, 15 y 30 s (centradas en cada frame)
base = ["L_cx", "L_sx", "L_sy", "L_linea_def", "L_vel",
        "V_cx", "V_sx", "V_sy", "V_linea_def", "V_vel"]
partes = [feats]
for nombre, n in {"5s": 125, "15s": 375, "30s": 751}.items():
    media = feats[base].rolling(n, center=True, min_periods=1).mean()
    partes.append(media.add_suffix(f"_m{nombre}"))
feats_v = pd.concat(partes, axis=1)
X_v = feats_v.iloc[idx]
print("Features con ventanas:", feats_v.shape)

# 2. Peso: cada tramo de cercanía al hueco suma lo mismo
t = pd.Series(np.asarray(tramos))
peso = (1 / t.map(t.value_counts()).astype(float)).values
peso = peso / peso.mean()

# 3. Evaluación fuera de muestra por bloques
def crear_lgbm():
    return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
                          random_state=0, n_jobs=1, deterministic=True,
                          force_row_wise=True, verbose=-1)

def evaluar(nombre, Xm, crear_modelo, w=None):
    prob = np.zeros(len(y))
    for b in range(6):
        train, test = bloque != b, bloque == b
        peso_b = None if w is None else w[train]
        modelo = crear_modelo().fit(Xm[train], y[train], sample_weight=peso_b)
        prob[test] = modelo.predict_proba(Xm[test])[:, 1]
    acierto = (prob > 0.5) == y
    fila = {"config": nombre, "acierto": acierto.mean(),
            "AUC": roc_auc_score(y, prob), "logloss": log_loss(y, prob)}
    for tr in ["0-2 s", "2-5 s", "5-10 s", ">10 s"]:
        fila[tr] = acierto[t.values == tr].mean()
    return fila

configs = [("A. actual (XGB)",            X,   modelo_xgb, None),
           ("B. + ventanas (XGB)",        X_v, modelo_xgb, None),
           ("C. + ventanas + peso (XGB)", X_v, modelo_xgb, peso),
           ("D. + ventanas + peso (LGBM)", X_v, crear_lgbm, peso)]

resultados = pd.DataFrame([evaluar(*c) for c in configs]).set_index("config")
print(resultados.round(3))

**Resultados.** Las ventanas solas empeoran (acierto 0.800, log-loss 0.484): son muchas features para frames tan parecidos entre sí, y una ventana larga puede mezclar dos posesiones. Con peso por tramo sí ayudan: mismo acierto general (0.812) y log-loss (0.457) que el modelo actual, pero mejor cerca de los huecos (0.753 en 0-2 s y 0.786 en 2-5 s). LightGBM queda por debajo de XGBoost en la misma configuración.

**Decisión.** Nos quedamos con XGBoost + ventanas + peso. La mejora es leve frente a la variación entre bloques, pero va en el caso que importa. Con esto cerramos la capa 2.

## 15. Probabilidad de posesión local en todo el partido
Entrenamos el modelo elegido con todas las semillas y calculamos P(local) para cada frame, incluidos los huecos del balón. En las semillas usamos la predicción fuera de muestra, para no darle al HMM probabilidades infladas. Esta es la salida de la capa 2.

In [ ]:
# Modelo final con todas las semillas
modelo_final = modelo_xgb().fit(X_v, y, sample_weight=peso)
p_local = 1 - modelo_final.predict_proba(feats_v)[:, 1]     # y=1 es visitante

# En las semillas, predicción fuera de muestra
prob_oof = np.zeros(len(y))
for b in range(6):
    train, test = bloque != b, bloque == b
    m = modelo_xgb().fit(X_v[train], y[train], sample_weight=peso[train])
    prob_oof[test] = m.predict_proba(X_v[test])[:, 1]
p_local[idx] = 1 - prob_oof

# Chequeo: cómo se reparte P(local) por zona de certeza
zona = np.select([control_claro, vis & ~control_claro, calificado & ~con_balon],
                 ["control claro", "visible dudoso", "sin balón"], "fuera")
resumen = (pd.DataFrame({"zona": zona, "p": p_local})[calificado]
           .groupby("zona")["p"]
           .describe(percentiles=[0.1, 0.5, 0.9])[["count", "mean", "10%", "50%", "90%"]])
print(resumen.round(3))
print("Frames con P(local) entre 0.4 y 0.6 (indecisos):",
      round(((p_local > 0.4) & (p_local < 0.6))[calificado].mean(), 3))

## 16. Calibración de las probabilidades
El modelo está demasiado seguro: sin balón casi siempre dice menos de 0.05 o más de 0.95, pero cerca de los huecos acierta ~75%. Si el HMM recibe probabilidades infladas, va a partir las posesiones con cada ruido.

Aplicamos *temperature scaling*: dividimos el logit de cada probabilidad por un número T elegido para minimizar el log-loss en las semillas a menos de 5 s de un hueco (predicciones fuera de muestra). T > 1 acerca las probabilidades a 0.5. Se ajusta con pseudo-etiquetas, sin usar el GT público.

In [ ]:
from scipy.optimize import minimize_scalar
from scipy.special import logit, expit

eps = 1e-6
p_semilla = p_local[idx]                      # fuera de muestra (paso 15)
real_local = (y == 0)
cerca = np.isin(t.values, ["0-2 s", "2-5 s"])

# Tabla de confiabilidad: ¿cuando dice X% de confianza, acierta X%?
conf = np.maximum(p_semilla, 1 - p_semilla)
acierto = (p_semilla > 0.5) == real_local
rangos = pd.cut(conf[cerca], [0.5, 0.6, 0.7, 0.8, 0.9, 1.0], include_lowest=True)
print(pd.DataFrame({"confianza": conf[cerca], "acierto": acierto[cerca]})
      .groupby(rangos, observed=True)
      .agg(semillas=("acierto", "size"), confianza=("confianza", "mean"), acierto=("acierto", "mean"))
      .round(3))

# Buscar la temperatura que minimiza el log-loss cerca de los huecos
z = logit(np.clip(p_semilla[cerca], eps, 1 - eps))
def perdida(T):
    return log_loss(real_local[cerca], expit(z / T), labels=[False, True])
T = minimize_scalar(perdida, bounds=(0.5, 10), method="bounded").x
print(f"\nTemperatura: {T:.2f} | log-loss cerca de huecos: {perdida(1):.3f} -> {perdida(T):.3f}")

# Aplicar a todo el partido
p_local_cal = expit(logit(np.clip(p_local, eps, 1 - eps)) / T)
resumen = (pd.DataFrame({"zona": zona, "p": p_local_cal})[calificado]
           .groupby("zona")["p"]
           .describe(percentiles=[0.1, 0.5, 0.9])[["mean", "10%", "50%", "90%"]])
print(resumen.round(3))
print("Indecisos (0.4-0.6):", round(((p_local_cal > 0.4) & (p_local_cal < 0.6))[calificado].mean(), 3))

**Resultados.** El modelo estaba demasiado seguro: cuando decía 95.6% acertaba 86.6%, y por debajo de 0.7 de confianza acierta como una moneda al aire. La temperatura óptima es T = 1.57 (log-loss cerca de huecos: 0.525 → 0.493). Después de calibrar, sin balón P(local) va de 0.10 a 0.86 (percentiles 10 y 90) y el 14% de los frames queda indeciso.

**Decisión.** Usamos `p_local_cal` como salida de la capa 2.

## 17. Capa 3: probabilidad de balón muerto
D corta posesiones (`H H D D H H` son 2), así que detectarlo bien es clave para spellF1. Usamos el mismo truco que en la capa 2: entrenamos con frames donde estamos seguros y propagamos al resto.

Positivos (D seguro): balón a más de 1 m fuera de la cancha y frames antes del primer saque inicial de cada tiempo. Negativos (juego vivo): las semillas. Las features no usan la posición del balón, para que el modelo funcione en los huecos: velocidad de los jugadores, forma de los equipos y tiempo hasta o desde un balón parado del Smart Tagging.

Límite: casi todos los positivos son balones fuera, así que las faltas dentro de la cancha están poco representadas.

In [ ]:
# Inicios de balón parado y de saques iniciales, con el cargador oficial
sys.path.insert(0, str(ROOT / "starter"))
from load_smart_tagging import load as cargar_smart_tagging

fases_st = cargar_smart_tagging()

def es_balon_parado(f):
    return (f["code"] or "").strip().lower() == "set pieces"

inicio_bp = np.array([f["start_frame"] for f in fases_st if es_balon_parado(f)])
inicio_ko = np.array([f["start_frame"] for f in fases_st if es_balon_parado(f)
                      and (f["tags"].get("Type") or "").strip().lower() == "kick off (start)"])

print("Fases:", len(fases_st), "| balón parado:", len(inicio_bp))
print("Saques de inicio de tiempo (frame de inicio de la etiqueta):", inicio_ko)

In [ ]:
# --- Positivos: balón fuera > 1 m y frames antes del primer saque inicial de cada tiempo ---
dist_fuera = np.maximum.reduce([-bal[:, 0], bal[:, 0] - LARGO, -bal[:, 1], bal[:, 1] - ANCHO])
balon_fuera = con_balon & (dist_fuera > 1)

inicio_ko = np.sort(np.asarray(inicio_ko))
antes_ko = np.zeros(N_FRAMES, bool)
for ini, fin in [(11185, 90652), (110808, 190198)]:
    ko_tiempo = inicio_ko[(inicio_ko >= ini) & (inicio_ko < fin)]
    if len(ko_tiempo):
        antes_ko[ini:ko_tiempo[0]] = True
        print(f"Tiempo desde {ini}: primer saque inicial en frame {ko_tiempo[0]}")

# --- Segundos hasta el próximo balón parado y desde el último (tope 60 s) ---
inicio_bp = np.sort(np.asarray(inicio_bp))
frames = np.arange(N_FRAMES)
pos = np.searchsorted(inicio_bp, frames)
proximo = np.where(pos < len(inicio_bp), inicio_bp[np.minimum(pos, len(inicio_bp) - 1)], np.inf)
anterior = np.where(pos > 0, inicio_bp[np.maximum(pos - 1, 0)], -np.inf)

def ventana(serie, n):
    return serie.rolling(n, center=True, min_periods=1).mean()

vel = feats[["L_vel", "V_vel"]].mean(axis=1)
fd = pd.DataFrame({
    "vel_3s": ventana(vel, 75),
    "vel_10s": ventana(vel, 250),
    "vel_min_3s": ventana(feats[["L_vel", "V_vel"]].min(axis=1), 75),
    "L_sx_3s": ventana(feats["L_sx"], 75), "L_sy_3s": ventana(feats["L_sy"], 75),
    "V_sx_3s": ventana(feats["V_sx"], 75), "V_sy_3s": ventana(feats["V_sy"], 75),
    "seg_a_bp": np.minimum((proximo - frames) / 25, 60),
    "seg_desde_bp": np.minimum((frames - anterior) / 25, 60),
})

# --- Entrenamiento con pseudo-etiquetas y validación por bloques ---
pos_d = (balon_fuera | antes_ko) & calificado
neg_d = control_claro & ~pos_d
idx_d = np.flatnonzero(pos_d | neg_d)
Xd, yd = fd.iloc[idx_d], pos_d[idx_d].astype(int)
bloque_d = pd.qcut(idx_d, 6, labels=False)
peso_d = np.where(yd == 1, (yd == 0).sum() / (yd == 1).sum(), 1.0)
print(f"\nPositivos D: {yd.sum()} | negativos: {(yd == 0).sum()}")

prob_d = np.zeros(len(yd))
for b in range(6):
    train, test = bloque_d != b, bloque_d == b
    m = modelo_xgb().fit(Xd[train], yd[train], sample_weight=peso_d[train])
    prob_d[test] = m.predict_proba(Xd[test])[:, 1]
print("AUC fuera de muestra:", round(roc_auc_score(yd, prob_d), 3))

# --- Modelo final y P(D) en todo el partido ---
modelo_d = modelo_xgb().fit(Xd, yd, sample_weight=peso_d)
p_muerto = modelo_d.predict_proba(fd)[:, 1]

# --- Chequeos de plausibilidad ---
print("\nP(D) media por zona:")
print(pd.Series(p_muerto[calificado]).groupby(zona[calificado]).mean().round(3))
es_d = p_muerto > 0.5
print("\n% de frames calificados con P(D) > 0.5:", round(es_d[calificado].mean(), 3))
duraciones = np.concatenate([huecos(es_d, ini, fin)[1] for ini, fin in [(11185, 90652), (110808, 190198)]])
print(f"Tramos de D: {len(duraciones)} | mediana {np.median(duraciones) / 25:.1f} s | p90 {np.percentile(duraciones, 90) / 25:.1f} s")

## 18. Capa 3 (v2): ¿qué huecos del balón son pausas?
El modelo frame a frame no funcionó: marcó solo 12.8% de D, contra ~40% esperable en fútbol, y en tramos de 1.3 s. Más de la mitad de sus positivos venía del entretiempo, que no se parece a una pausa del partido.

Cambiamos de enfoque: el tracker pierde el balón cuando sale del juego, así que clasificamos cada hueco. Es pausa si el balón se perdió o reapareció a menos de 3 m de una línea, si reapareció en el centro (saque tras gol) o si hay un inicio de balón parado del Smart Tagging dentro del hueco o hasta 5 s después. Sumamos los balones fuera visibles y el tramo antes del saque del segundo tiempo.

Validación sin GT: la velocidad de los jugadores no entra en la regla, así que si los huecos marcados como pausa son más lentos, eso confirma la regla de forma independiente.

In [ ]:
tiempos = [(11185, 90652), (110808, 190198)]

# Lista de huecos [inicio, fin) en frames calificados
lista = []
for ini, fin in tiempos:
    comienzos, largos = huecos(~con_balon, ini, fin)
    lista += [(a, a + n) for a, n in zip(comienzos, largos)]

def dist_borde(i):
    if i < 0 or i >= N_FRAMES or not con_balon[i]:
        return np.nan
    x, y = bal[i, 0], bal[i, 1]
    return min(x, LARGO - x, y, ANCHO - y)

def dist_centro(i):
    if i < 0 or i >= N_FRAMES or not con_balon[i]:
        return np.nan
    return np.hypot(bal[i, 0] - LARGO / 2, bal[i, 1] - ANCHO / 2)

g = pd.DataFrame([{
    "ini": a, "fin": b, "dur_s": (b - a) / 25,
    "d_salida": dist_borde(a - 1),          # dónde se perdió el balón
    "d_entrada": dist_borde(b),             # dónde reapareció
    "entra_centro": dist_centro(b) < 3,
    "reinicio": np.any((inicio_bp >= a) & (inicio_bp < b + 125)),
    "vel": np.nanmean(vel.values[a:b]),
} for a, b in lista])

def marcar_pausas(umbral):
    return ((g.d_salida < umbral) | (g.d_entrada < umbral) | g.entra_centro | g.reinicio).values

def mascara_muerto(es_pausa):
    m = balon_fuera | antes_ko
    for (a, b), p in zip(lista, es_pausa):
        if p:
            m[a:b] = True
    return m & calificado

# Sensibilidad al umbral de distancia a la línea
for umbral in [1.5, 3, 5]:
    p = marcar_pausas(umbral)
    print(f"Umbral {umbral} m: huecos pausa {p.sum()}/{len(p)} | "
          f"% D en frames calificados {mascara_muerto(p)[calificado].mean():.3f}")

# Detalle con 3 m
es_pausa = marcar_pausas(3)
muerto = mascara_muerto(es_pausa)
print("\nMotivos (un hueco puede tener varios):")
print("  se perdió cerca de una línea:", (g.d_salida < 3).sum())
print("  reapareció cerca de una línea:", (g.d_entrada < 3).sum())
print("  reapareció en el centro:", g.entra_centro.sum())
print("  inicio de balón parado:", g.reinicio.sum())

print("\nValidación independiente (velocidad media de jugadores, m/s):")
print("  huecos pausa:     ", round(g.vel[es_pausa].mean(), 2))
print("  huecos juego vivo:", round(g.vel[~es_pausa].mean(), 2))
print("  duración mediana (s): pausa", g.dur_s[es_pausa].median(), "| vivo", g.dur_s[~es_pausa].median())

dur = np.concatenate([huecos(muerto, ini, fin)[1] for ini, fin in tiempos])
print(f"\nTramos de D: {len(dur)} | mediana {np.median(dur) / 25:.1f} s | p90 {np.percentile(dur, 90) / 25:.1f} s")

**Resultados.** 79 de 150 huecos son pausas, y el % de D queda en 31.2% (30.7% con 1.5 m y 33.0% con 5 m: es estable). Los huecos de pausa son más lentos que los de juego vivo (1.32 contra 1.78 m/s) y más largos (15.2 contra 7.1 s de mediana), lo que confirma la regla sin haber usado la velocidad. Quedan 97 tramos de D con mediana de 11.7 s.

**Decisión.** Esta máscara es la evidencia de D para el HMM, y el modelo frame a frame se descarta. Pendiente: las pausas con balón visible (faltas), que explican por qué queda por debajo del ~40% esperable.

## 19. Capa 4: HMM + Viterbi
Tres estados: H, A y D. La evidencia por frame viene de las capas anteriores: la máscara de pausas para D, el más cercano en las semillas, y el modelo de forma calibrado (combinado con el más cercano cuando el balón se ve pero hay duda). Viterbi busca la secuencia que mejor explica toda la evidencia, pagando un costo λ por cada cambio de equipo: un frame suelto no alcanza para cambiar de posesión, hace falta evidencia sostenida.

Como frames seguidos son casi idénticos, 25 frames (1 s) cuentan como una sola observación. λ se elige por plausibilidad (número y duración de las posesiones), no con el GT público.

In [ ]:
H, A, D = 0, 1, 2
tiempos = [(11185, 90652), (110808, 190198)]

# --- Evidencia de equipo: P(H) ---
p_h = p_local_cal.copy()
p_h[control_claro] = np.where(equipo_cercano[control_claro] == 0, 0.9, 0.1)
dudoso = vis & ~control_claro
q = np.where(equipo_cercano == 0, 0.6, 0.4)                  # pista débil del más cercano
combinado = p_local_cal * q / (p_local_cal * q + (1 - p_local_cal) * (1 - q))
p_h[dudoso] = combinado[dudoso]

# --- Evidencia de balón muerto: P(D) ---
p_d = np.where(muerto, 0.95, 0.03)

# --- Log-evidencia por frame y estado (25 frames = 1 observación) ---
eps = 1e-6
log_e = np.log(np.clip(np.column_stack([(1 - p_d) * p_h, (1 - p_d) * (1 - p_h), p_d]), eps, 1)) / 25

def viterbi(log_e, trans):
    T = len(log_e)
    score = log_e[0].copy()
    atras = np.zeros((T, 3), np.int8)
    for t in range(1, T):
        cand = score[:, None] + trans
        atras[t] = cand.argmax(axis=0)
        score = cand.max(axis=0) + log_e[t]
    camino = np.empty(T, np.int8)
    camino[-1] = score.argmax()
    for t in range(T - 1, 0, -1):
        camino[t - 1] = atras[t, camino[t]]
    return camino

def segmentar(lam_cambio, lam_d=1.0):
    trans = np.array([[0, -lam_cambio, -lam_d],
                      [-lam_cambio, 0, -lam_d],
                      [-lam_d, -lam_d, 0]], float)
    etq = np.full(N_FRAMES, D, np.int8)                        # frames no calificados: D
    for ini, fin in tiempos:
        etq[ini:fin] = viterbi(log_e[ini:fin], trans)
    return etq

def describir(etq):
    valores, largos, cambios_directos = [], [], 0
    for ini, fin in tiempos:
        e = etq[ini:fin]
        inicios = np.r_[0, np.flatnonzero(e[1:] != e[:-1]) + 1]
        v = e[inicios]
        valores.append(v)
        largos.append(np.diff(np.r_[inicios, len(e)]))
        cambios_directos += np.sum((v[:-1] != D) & (v[1:] != D))
    v, l = np.concatenate(valores), np.concatenate(largos)
    vivo = v != D
    e_cal = etq[calificado]
    return {"posesiones": int(vivo.sum()),
            "media_s": l[vivo].mean() / 25, "mediana_s": np.median(l[vivo]) / 25,
            "cambios_H_A_directos": int(cambios_directos),
            "pct_D": (e_cal == D).mean(),
            "pct_H_del_vivo": (e_cal == H).sum() / (e_cal != D).sum()}

tabla = pd.DataFrame({lam: describir(segmentar(lam)) for lam in [1, 2, 3, 5, 8]}).T
tabla.index.name = "λ cambio"
print(tabla.round(3))

In [ ]:
etiquetas = segmentar(3)
sub = pd.DataFrame({"frame": np.arange(N_FRAMES), "label": np.array(["H", "A", "D"])[etiquetas]})
salida = ROOT / "evaluation" / "results" / "javier_v1.csv"
sub.to_csv(salida, index=False)
print("Guardado en:", salida)

**Resultados v1.** Con λ ≥ 3 no hubo ni un cambio directo de H a A: Viterbi encontró un atajo. Pasar por D (H → D → A) costaba 2, menos que cambiar directo (λ = 3), así que metía un frame de D falso en cada cambio de posesión. Por eso λ dejaba de importar a partir de 2.

**Decisión.** La máscara de pausas pasa a ser una restricción dura: esos frames son D y no se discuten. Dentro de cada tramo de juego vivo, Viterbi elige solo entre H y A, con un costo λ por cada cambio de equipo.

In [ ]:
# Evidencia H/A por frame (25 frames = 1 observación)
log_e2 = np.log(np.clip(np.column_stack([p_h, 1 - p_h]), eps, 1)) / 25

def viterbi(log_e, trans):
    T, K = log_e.shape
    score = log_e[0].copy()
    atras = np.zeros((T, K), np.int8)
    for t in range(1, T):
        cand = score[:, None] + trans
        atras[t] = cand.argmax(axis=0)
        score = cand.max(axis=0) + log_e[t]
    camino = np.empty(T, np.int8)
    camino[-1] = score.argmax()
    for t in range(T - 1, 0, -1):
        camino[t - 1] = atras[t, camino[t]]
    return camino

def segmentar(lam_cambio):
    trans = np.array([[0, -lam_cambio], [-lam_cambio, 0]], float)
    etq = np.full(N_FRAMES, D, np.int8)                  # pausas y no calificados: D
    for ini, fin in tiempos:
        comienzos, largos = huecos(~muerto, ini, fin)    # tramos de juego vivo
        for a, n in zip(comienzos, largos):
            etq[a:a + n] = viterbi(log_e2[a:a + n], trans)
    return etq

tabla = pd.DataFrame({lam: describir(segmentar(lam)) for lam in [0.5, 1, 2, 3, 5, 8]}).T
tabla.index.name = "λ cambio"
print(tabla.round(3))

**Resultados v2.** Sin el atajo por D, λ controla de verdad los cambios: de 370 posesiones (λ = 0.5) a 144 (λ = 8), con el % de D fijo en 31.2%. No hay un codo claro. Las pausas solas ya cortan el partido en ~96 tramos de juego vivo, que es el piso de posesiones.

**Decisión.** λ = 2: 232 posesiones de 18.8 s de media, dentro del rango plausible (150-250 posesiones de 10-20 s). Como el % de D está subestimado, las duraciones reales serían algo menores, y eso descarta λ más altos. Es una elección por plausibilidad: se debe confirmar con un partido externo.

In [ ]:
etiquetas = segmentar(2)
sub = pd.DataFrame({"frame": np.arange(N_FRAMES), "label": np.array(["H", "A", "D"])[etiquetas]})
salida = ROOT / "evaluation" / "results" / "javier_v2.csv"
sub.to_csv(salida, index=False)
print("Guardado en:", salida)

## 20. Primera entrega (v2)
`javier_v2.csv` pasa `validate_submission`: 232 posesiones de 18.8 s de media, D 31.2% de los frames calificados.

Chequeo de pipeline contra el GT público (5 min, 17 posesiones; no se usa para elegir nada): spellF1 0.579, contra 0.326 de baseline_smooth50. En el partido completo se espera bastante menos (smooth50 baja de 0.326 a 0.243). Esta es la entrega de seguridad. Lo siguiente es validar con un partido externo y mejorar la detección de pausas con el balón visible.

In [ ]:
import numpy as np, pandas as pd

tiempos = [(11185, 90652), (110808, 190198)]
a = pd.read_csv(r"C:\Users\jfagu\Documents\Proyectos\mtah2027\mtah-posesion\evaluation\results\javier_v2.csv")["label"].values
b = pd.read_csv(r"C:\Users\jfagu\Downloads\submission_final.csv")["label"].values

cal = np.zeros(len(a), bool)
for ini, fin in tiempos:
    cal[ini:fin] = True

def llenar_x(x):
    """Igual que el scorer: cada X toma la última etiqueta (y al inicio del tiempo, la primera)."""
    x = x.copy()
    for ini, fin in tiempos:
        e = pd.Series(x[ini:fin]).replace("X", np.nan).ffill().bfill()
        x[ini:fin] = e.fillna("X").values
    return x

print("X en frames calificados | mío:", round((a[cal] == "X").mean(), 3),
      "| suyo:", round((b[cal] == "X").mean(), 3))
a, b = llenar_x(a), llenar_x(b)

print("\nCoinciden en frames calificados:", round((a[cal] == b[cal]).mean(), 3))
print(pd.crosstab(a[cal], b[cal], rownames=["mío"], colnames=["suyo"]))

def resumen(x):
    n = dur = 0
    for ini, fin in tiempos:
        e = x[ini:fin]
        cortes = np.r_[0, np.flatnonzero(e[1:] != e[:-1]) + 1, len(e)]
        for i, j in zip(cortes[:-1], cortes[1:]):
            if e[i] in ("H", "A"):
                n += 1
                dur += j - i
    return n, round(dur / n / 25, 1)

print("\nPosesiones y duración media (s) | mío:", resumen(a), "| suyo:", resumen(b))
print("% de D en calificados | mío:", round((a[cal] == "D").mean(), 3),
      "| suyo:", round((b[cal] == "D").mean(), 3))

In [ ]:
from pathlib import Path

ROOT = Path.cwd()                      # carpeta del repo (mtah-posesion)
base = ROOT.parent / "comparacion"     # fuera del repo
print("Repo:", ROOT)
print("Comparación irá en:", base)

In [ ]:
from pathlib import Path

base = ROOT.parent / "comparacion"                      # fuera del repo
for carpeta in ["cand_javier", "cand_companero", "salida_1", "salida_2"]:
    (base / carpeta).mkdir(parents=True, exist_ok=True)

a0 = pd.read_csv(r"C:\Users\jfagu\Documents\Proyectos\mtah2027\mtah-posesion\evaluation\results\javier_v2.csv")["label"].values
b0 = pd.read_csv(r"C:\Users\jfagu\Downloads\submission_final.csv")["label"].values

def como_referencia(x):
    x = llenar_x(x).copy()
    x[~cal] = "X"                                       # solo cuentan los frames calificados
    return x

def guardar(x, ruta):
    pd.DataFrame({"frame": np.arange(len(x)), "label": x}).to_csv(ruta, index=False)

guardar(como_referencia(a0), base / "ref_javier.csv")
guardar(como_referencia(b0), base / "ref_companero.csv")
guardar(a0, base / "cand_javier" / "javier_v2.csv")
guardar(b0, base / "cand_companero" / "companero_v1.csv")

r1, c1, o1 = base / "ref_javier.csv", base / "cand_companero", base / "salida_1"
r2, c2, o2 = base / "ref_companero.csv", base / "cand_javier", base / "salida_2"
print(f'python evaluation/run_evaluation.py --gt "{r1}" --results "{c1}" --out "{o1}" --bootstrap 0')
print(f'python evaluation/run_evaluation.py --gt "{r2}" --results "{c2}" --out "{o2}" --bootstrap 0')